In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Tkinter GUI for concatenating PDFs.

Dependencies:
    pip install pypdf   # PyPDF2 has been renamed to pypdf in recent releases
"""

import os
import sys
import tkinter as tk
from tkinter import filedialog, messagebox

try:
    from pypdf import PdfMerger  # PyPDF2>=3.0 uses the same API under 'pypdf'
except ImportError:  # fallback to older names
    try:
        from PyPDF2 import PdfFileMerger as PdfMerger
    except Exception as e:
        print("Could not import any PDF merger library.")
        raise e

# --------------------------------------------------------------------------- #
# Helper functions
# --------------------------------------------------------------------------- #

def merge_pdfs(pdf_paths, output_path):
    """
    Merge a list of PDFs into a single file.

    Parameters
    ----------
    pdf_paths : List[str]
        Paths to input PDFs.
    output_path : str
        Path where the merged PDF will be written.
    """
    merger = PdfMerger()

    for path in pdf_paths:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"File does not exist: {path}")
        try:
            merger.append(path)
        except Exception as e:
            raise RuntimeError(f"Could not read PDF '{path}': {e}")

    # Write the merged file
    try:
        with open(output_path, 'wb') as fout:
            merger.write(fout)
    finally:
        merger.close()


# --------------------------------------------------------------------------- #
# Tkinter GUI
# --------------------------------------------------------------------------- #

class PdfMergerGUI(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("PDF Concatenator")
        self.geometry("500x250")
        self.resizable(False, False)

        # Variables to hold user selections
        self.pdf_paths = []  # List of selected PDFs
        self.output_path = ""  # Path for the merged PDF

        # UI widgets
        self._create_widgets()

    def _create_widgets(self):
        padding = {'padx': 10, 'pady': 5}

        # Select PDFs button
        self.select_btn = tk.Button(
            self,
            text="Select PDF Files",
            command=self.select_pdfs,
            width=25
        )
        self.select_btn.grid(row=0, column=0, **padding)

        # Label to show count of selected files
        self.count_label = tk.Label(self, text="No files selected")
        self.count_label.grid(row=0, column=1, sticky='w')

        # Output file selection button
        self.output_btn = tk.Button(
            self,
            text="Choose Output PDF",
            command=self.select_output_file,
            width=25
        )
        self.output_btn.grid(row=1, column=0, **padding)

        # Label to show chosen output path (shortened)
        self.output_label = tk.Label(self, text="No output file selected")
        self.output_label.grid(row=1, column=1, sticky='w')

        # Merge button
        self.merge_btn = tk.Button(
            self,
            text="Merge PDFs",
            command=self.start_merge,
            width=25,
            bg='#4CAF50',
            fg='white'
        )
        self.merge_btn.grid(row=2, column=0, **padding)

        # Quit button
        self.quit_btn = tk.Button(
            self,
            text="Quit",
            command=self.destroy,
            width=25,
            bg='#f44336',
            fg='white'
        )
        self.quit_btn.grid(row=2, column=1, **padding)

    def select_pdfs(self):
        files = filedialog.askopenfilenames(
            title="Select PDF files to merge",
            filetypes=[("PDF Files", "*.pdf")],
            multiple=True
        )
        if files:
            # Store as list of strings (paths)
            self.pdf_paths = list(files)
            self.count_label.config(text=f"{len(self.pdf_paths)} file(s) selected")

    def select_output_file(self):
        out_path = filedialog.asksaveasfilename(
            title="Save Merged PDF As",
            defaultextension=".pdf",
            filetypes=[("PDF Files", "*.pdf")],
            initialfile="merged.pdf"
        )
        if out_path:
            self.output_path = out_path
            display_name = os.path.basename(out_path)
            self.output_label.config(text=display_name)

    def start_merge(self):
        # Basic validation
        if not self.pdf_paths:
            messagebox.showwarning("No input files", "Please select at least one PDF file.")
            return

        if not self.output_path:
            messagebox.showwarning("No output file", "Please choose an output PDF path.")
            return

        try:
            merge_pdfs(self.pdf_paths, self.output_path)
        except Exception as e:
            messagebox.showerror("Merge failed", f"An error occurred while merging:\n{e}")
            return

        messagebox.showinfo(
            "Success",
            f"Successfully merged {len(self.pdf_paths)} PDF(s) into:\n{self.output_path}"
        )
        # Reset state for convenience
        self.pdf_paths = []
        self.output_path = ""
        self.count_label.config(text="No files selected")
        self.output_label.config(text="No output file selected")


# --------------------------------------------------------------------------- #
# Main entry point
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    app = PdfMergerGUI()
    app.mainloop()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
PDF Merger GUI
==============

This small Tkinter app wraps a shell script called `merge_pdf.sh`
which expects an output PDF name followed by any number of input PDFs.
The GUI allows you to:

* Browse for PDFs and add them to the list
* Remove selected PDFs from the list
* Move PDFs up / down in the order
* Specify the output filename (default: merged.pdf)
* Execute the merge script and show the status

Prerequisites
-------------
* Python 3.6+
* Tkinter (comes with standard CPython)
* A Unix‑like environment where `merge_pdf.sh` is executable.
"""

import os
import sys
import subprocess
from pathlib import Path
from tkinter import (
    Tk,
    Button,
    Entry,
    Label,
    Listbox,
    Scrollbar,
    StringVar,
    END,
    SINGLE,
    Toplevel,
    filedialog,
    messagebox,
)

# --------------------------------------------------------------------------- #
# Helper functions
# --------------------------------------------------------------------------- #

def run_merge_script(output_name: str, input_files: list[str]) -> tuple[int, str]:
    """
    Execute `merge_pdf.sh` with the given arguments.

    Parameters
    ----------
    output_name : str
        Name of the resulting PDF.
    input_files : list[str]
        List of PDFs to merge (in order).

    Returns
    -------
    returncode : int
        The script's exit code.
    stdout_stdout : str
        Combined stdout & stderr for debugging.
    """
    # Build command: ./merge_pdf.sh output file1 file2 ...
    cmd = ["./merge_pdf.sh", output_name] + input_files

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=False,  # we handle non‑zero ourselves
            cwd=os.getcwd(),   # run in current working dir
            shell=False,
        )
        return result.returncode, result.stdout + "\n" + result.stderr
    except FileNotFoundError:
        return -1, f"Script not found: {cmd[0]}"
    except Exception as e:
        return -1, f"Unexpected error: {e}"


# --------------------------------------------------------------------------- #
# GUI Class
# --------------------------------------------------------------------------- #

class PdfMergerApp(Tk):
    def __init__(self):
        super().__init__()
        self.title("PDF Merger")
        self.geometry("600x400")
        self.resizable(False, False)

        # --- Widgets ----------------------------------------------------- #
        # Listbox with scrollbar
        lb_frame = Toplevel(self)  # use separate frame for better layout
        lb_frame.wm_title("Selected PDFs")
        lb_frame.geometry("500x250")

        self.lb_scroll = Scrollbar(lb_frame)
        self.lb_pdf = Listbox(
            lb_frame,
            selectmode=SINGLE,
            width=70,
            yscrollcommand=self.lb_scroll.set,
        )
        self.lb_scroll.config(command=self.lb_pdf.yview)

        self.lb_pdf.pack(side="left", fill="both", expand=True, padx=(10, 0), pady=10)
        self.lb_scroll.pack(side="right", fill="y", padx=(0, 10), pady=10)

        # Buttons for manipulating the list
        btn_frame = Toplevel(self)
        btn_frame.wm_title("Controls")
        btn_frame.geometry("500x100")

        Button(btn_frame, text="+ Add PDF", command=self.add_pdf).pack(
            side="left", padx=5, pady=10
        )
        Button(btn_frame, text="- Remove", command=self.remove_pdf).pack(
            side="left", padx=5, pady=10
        )
        Button(btn_frame, text="↑ Move Up", command=lambda: self.move_pdf(-1)).pack(
            side="left", padx=5, pady=10
        )
        Button(btn_frame, text="↓ Move Down", command=lambda: self.move_pdf(1)).pack(
            side="left", padx=5, pady=10
        )

        # Output filename entry
        output_lbl = Label(self, text="Output PDF:")
        output_lbl.pack(pady=(20, 0))
        self.output_var = StringVar(value="merged.pdf")
        self.entry_output = Entry(self, width=30, textvariable=self.output_var)
        self.entry_output.pack()

        # Merge button
        self.btn_merge = Button(
            self,
            text="Merge PDFs",
            command=self.merge_pdfs,
            state="disabled",
            width=20,
        )
        self.btn_merge.pack(pady=10)

        # Status label
        self.status_var = StringVar()
        Label(self, textvariable=self.status_var, fg="blue").pack()

        # Bind listbox selection to enable/disable merge button
        self.lb_pdf.bind("<<ListboxSelect>>", lambda e: self.update_merge_button())

    # --------------------------------------------------------------------- #
    # UI callbacks
    # --------------------------------------------------------------------- #

    def add_pdf(self):
        """Open file dialog and add selected PDFs to the Listbox."""
        files = filedialog.askopenfilenames(
            title="Choose PDF(s) to merge",
            filetypes=[("PDF Files", "*.pdf")],
            initialdir=str(Path.home()),
        )
        for f in files:
            if f not in self.lb_pdf.get(0, END):
                self.lb_pdf.insert(END, f)
        self.update_merge_button()

    def remove_pdf(self):
        """Remove selected item(s) from the Listbox."""
        selection = self.lb_pdf.curselection()
        if selection:
            index = selection[0]
            self.lb_pdf.delete(index)
        self.update_merge_button()

    def move_pdf(self, direction: int):
        """
        Move the selected PDF up or down in the list.

        Parameters
        ----------
        direction : int
            -1 to move up, 1 to move down.
        """
        index = self.lb_pdf.curselection()
        if not index:
            return
        idx = index[0]
        new_index = idx + direction

        if new_index < 0 or new_index >= self.lb_pdf.size():
            return  # can't move out of bounds

        item_text = self.lb_pdf.get(idx)
        self.lb_pdf.delete(idx)
        self.lb_pdf.insert(new_index, item_text)
        self.lb_pdf.select_set(new_index)

    def update_merge_button(self):
        """Enable Merge button only if we have at least one PDF and an output name."""
        has_pdfs = self.lb_pdf.size() > 0
        out_name = self.output_var.get().strip()
        self.btn_merge.config(state="normal" if (has_pdfs and out_name) else "disabled")

    def merge_pdfs(self):
        """Run the merge script with selected PDFs."""
        output_name = self.output_var.get().strip()
        if not output_name:
            messagebox.showwarning("Missing Output", "Please specify an output file name.")
            return

        input_files = list(self.lb_pdf.get(0, END))

        # Disable button while running
        self.btn_merge.config(state="disabled")
        self.status_var.set("Running merge…")

        code, msg = run_merge_script(output_name, input_files)

        if code == 0:
            self.status_var.set(f"✅ Merged successfully → {output_name}")
            messagebox.showinfo("Success", f"Merged PDF written to: {output_name}")
        else:
            self.status_var.set("❌ Merge failed")
            messagebox.showerror(
                "Merge Failed",
                f"Error code: {code}\n\n{msg}",
            )

        # Re‑enable button
        self.update_merge_button()


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    # Ensure the script exists and is executable
    script_path = Path("./merge_pdf.sh")
    if not script_path.exists():
        messagebox.showerror(
            "Missing Script",
            f"The required script '{script_path}' was not found in the current directory.\n"
            "Make sure it exists and has execute permissions.",
        )
        sys.exit(1)
    if not os.access(script_path, os.X_OK):
        # Try to make it executable
        try:
            os.chmod(script_path, 0o755)
        except Exception as e:
            messagebox.showerror(
                "Permission Error",
                f"Cannot set execute permission on '{script_path}': {e}",
            )
            sys.exit(1)

    app = PdfMergerApp()
    app.mainloop()


In [ ]:
#!/usr/bin/env python3
"""
Merge PDFs using an existing shell script via a Tkinter GUI.

Command line that will ultimately be executed:
    ./merge_pdf.sh all_pages.pdf chapter1.pdf chapter2.pdf appendix.pdf

The GUI lets you:
    • Choose the output PDF name (first argument to the script)
    • Add any number of input PDF files
    • Click “Merge” and see a status message.
"""

import os
import subprocess
import sys
from pathlib import Path
from tkinter import (
    Tk, Button, Listbox, Entry, Label,
    END, SINGLE, MULTIPLE, DISABLED, NORMAL,
    filedialog, messagebox
)

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #

SCRIPT_NAME = "merge_pdf.sh"           # the script that does the merging
SCRIPT_DIR  = Path(__file__).parent   # folder containing this .py file
SCRIPT_PATH = SCRIPT_DIR / SCRIPT_NAME

# --------------------------------------------------------------------------- #
# Helper functions
# --------------------------------------------------------------------------- #

def run_merge(output_file: str, input_files: list[str]) -> None:
    """
    Execute the merge_pdf.sh script with the given filenames.
    Raises subprocess.CalledProcessError on failure.
    """
    # Build command: ["./merge_pdf.sh", output.pdf, in1.pdf, in2.pdf, …]
    cmd = [str(SCRIPT_PATH), output_file] + input_files
    print("Running command:", " ".join(cmd))
    subprocess.run(cmd, check=True)  # will raise if non‑zero exit code


# --------------------------------------------------------------------------- #
# Tkinter Application class
# --------------------------------------------------------------------------- #

class MergeApp:
    def __init__(self, root: Tk):
        self.root = root
        root.title("PDF Merger")
        root.resizable(False, False)

        # ----- Output file -----
        Label(root, text="Output PDF:").grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.out_entry = Entry(root, width=40)
        self.out_entry.grid(row=0, column=1, padx=5, pady=5)

        Button(root, text="Browse…", command=self.choose_output).grid(
            row=0, column=2, padx=5, pady=5
        )

        # ----- Input files list -----
        Label(root, text="Input PDFs:").grid(row=1, column=0, sticky='nw', padx=5, pady=5)
        self.listbox = Listbox(
            root,
            width=55,
            height=8,
            selectmode=MULTIPLE
        )
        self.listbox.grid(row=1, column=1, columnspan=2, padx=5, pady=5)

        Button(root, text="Add PDFs…", command=self.add_files).grid(
            row=2, column=1, sticky='w', padx=5, pady=5
        )
        Button(root, text="Remove Selected", command=self.remove_selected).grid(
            row=2, column=1, sticky='e', padx=5, pady=5
        )

        # ----- Merge button -----
        self.merge_btn = Button(root, text="Merge PDFs", command=self.start_merge)
        self.merge_btn.grid(row=3, column=0, columnspan=3, pady=10)

    # ----------------------------------------------------------------------- #
    # UI callbacks
    # ----------------------------------------------------------------------- #

    def choose_output(self):
        """Open a Save‑As dialog to pick the output PDF name."""
        file_path = filedialog.asksaveasfilename(
            title="Select Output PDF",
            defaultextension=".pdf",
            filetypes=[("PDF files", "*.pdf")],
        )
        if file_path:
            self.out_entry.delete(0, END)
            self.out_entry.insert(0, file_path)

    def add_files(self):
        """Open a File dialog to pick one or more PDFs to merge."""
        paths = filedialog.askopenfilenames(
            title="Select PDF Files",
            filetypes=[("PDF files", "*.pdf")],
        )
        for path in paths:
            if path not in self.listbox.get(0, END):
                self.listbox.insert(END, path)

    def remove_selected(self):
        """Remove the currently selected items from the listbox."""
        selected = list(self.listbox.curselection())
        for index in reversed(selected):  # reverse to avoid shifting indices
            self.listbox.delete(index)

    def start_merge(self):
        """Validate inputs and run the merge script in a subprocess."""
        out_file = self.out_entry.get().strip()
        if not out_file:
            messagebox.showerror("Error", "Please specify an output PDF file.")
            return

        input_files = list(self.listbox.get(0, END))
        if not input_files:
            messagebox.showerror("Error", "Add at least one input PDF file to merge.")
            return

        # Disable UI during processing
        self.merge_btn.config(state=DISABLED)
        try:
            run_merge(out_file, input_files)
            messagebox.showinfo(
                "Success",
                f"PDFs merged successfully into:\n{out_file}"
            )
        except subprocess.CalledProcessError as exc:
            # Capture stderr from the script if available
            msg = (
                f"An error occurred while merging PDFs.\n"
                f"Command exited with status {exc.returncode}."
            )
            messagebox.showerror("Merge Failed", msg)
        finally:
            self.merge_btn.config(state=NORMAL)


# --------------------------------------------------------------------------- #
# Main entry point
# --------------------------------------------------------------------------- #

def main():
    # Basic sanity check: is the merge script present and executable?
    if not SCRIPT_PATH.is_file() or not os.access(SCRIPT_PATH, os.X_OK):
        print(f"Error: Merge script '{SCRIPT_PATH}' not found or not executable.")
        sys.exit(1)

    root = Tk()
    app = MergeApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()
